# API - Requests + BytesIO

In [ ]:
import csv
import io
import requests
from collections import defaultdict

url = "https://fsu.edu"
response = requests.get(url) # requests.get(url,stream=True) + response.iter_lines()
response.raise_for_status()

# writes the data to an in-memory binary byte stream, allowing us to process data line-by-line (instead of loading the entire payload to memory, like with response.json())
file_stream = io.BytesIO(response.content)

# TextIOWrapper wraps binary stream to decode raw bytes into readable string
    # process large datasets chunk-by-chunk from disk to avoid loading everything into memory at once
text_stream = io.TextIOWrapper(file_stream, encoding="utf-8")

reader = csv.DictReader(text_stream)

total_rows = 0
state_counts = defaultdict(int)

for row in reader:
    total_rows += 1
    state = row.get("State", "")
    state_counts[state] += 1


In [ ]:
# -----------------------------------------

# In-Memory ETL Blueprint

In [ ]:
import csv
import io
import json
import requests

# Writing to Files

In [ ]:
# JSON
with open("../output/users.json", mode="w", encoding="utf-8") as f:
    json.dump(transformed_data, f, indent=4)

In [ ]:
# JSONL
with open("../output/users.jsonl", mode="w", encoding="utf-8") as f:
    for line in transformed_data:
        f.write(json.dumps(record) + "\n")

In [ ]:
# CSV
headers = list(transformed_data[0].keys())

with open("../output/users.csv", mode="w", newline='', encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=headers)
    writer.writeheader()
    writer.writerows(transformed_data)

# ETL Pipeline - OOP

In [ ]:
import csv
import io
import json
import requests

In [ ]:
class BaseETLPipeline:
    def __init__(self, url, data_format="json"):
        self.url = url
        self.data_format = data_format.lower()
        self.raw_data = []
        self.transformed_data = []

    # fetches binary data from the network into an in-memory buffer
    def extract(self):
        response = requests.get(self.url)
        response.raise_for_status()
        return io.BytesIO(response.content) # content = bytes

    def _clean_row(self, row):
        return { k.strip().lower(): v.strip() for k, v in row.items() }

    def transform(self, buffer):
        text_stream = io.TextIOWrapper(buffer, encoding="utf-8")

        if self.data_format == "json":
            self.raw_data = json.load(text_stream)
        elif self.data_format == "csv":
            reader = csv.DictReader(text_stream)
            self.raw_data = [self._clean_row(row) for row in reader]

        for record in self.raw_data:
            try:
                if "price" not in record or "quantity" not in record:
                    continue

                category = record.get("category", "Unknown").strip().title()
                price = float(record.get("price"))
                quantity = int(record.get("quantity"))

                if price <= 0 or quantity <= 0:
                    continue

                self.transformed_data.append(
                    {
                        "category": category,
                        "total_revenue": price*quantity
                    }
                )

            except (ValueError, TypeError) as e:
                print(f"Error occurred: {e}")
                continue

    def aggregate(self):
        results = {}
        for item in self.transformed_data:
            cat = item["category"]
            rev = item["total_revenue"]
            results[cat] = results.get(cat, 0.0) + rev
        return results

    def run(self):
        data_buffer = self.extract()
        self.transform(data_buffer)
        return self.aggregate()


# Top K

In [6]:
import io
import logging
from collections import Counter

logging.basicConfig(level=logging.ERROR)

In [ ]:
def get_top_k_zipcodes(log_stream, k):
    zipcode_counts = Counter() # stream directly to this, so we never hold the whole list in memory

    for line in log_stream:
        cleaned_line = line.strip()
        if not cleaned_line:
            continue

        try:
            parts = [p.strip() for p in cleaned_line.split(",")]
            if len(parts) != 2:
                raise ValueError("Invalid row format")
            
            property_id, zip_code = parts

            if not zip_code.isdigit() or len(zip_code) != 5:
                raise ValueError(f"Malformed zipcode: {zip_code}")
            
            zipcode_counts[zip_code] += 1

        except (ValueError, Exception) as e:
            logging.error(f"Skipping row, error: {e}")
            continue

    return zipcode_counts.most_common(k)

In [ ]:
if __name__ == "__main__":
    mock_stream_buffer = io.StringIO("""
    prop_1,98101
    prop_2,98101
    prop_3,98111
    prop_4,4
    prop_5,77777
    prop_6,BAD
    prop_7,94103
    prop_8,98101
    prop_9,94103
    prop_10,98111
    """)

    stream = (line for line in mock_stream_buffer) # create generator stream via Generator Expression

    top_4_zips = get_top_k_zipcodes(stream, k=4)
    print(top_4_zips)

ERROR:root:Skipping row, error: Malformed zipcode: 4
ERROR:root:Skipping row, error: Malformed zipcode: BAD


[('98101', 3), ('98111', 2), ('94103', 2), ('77777', 1)]


# StringIO vs BytesIO vs TextIOWrapper

In [ ]:
# StringIO and BytesIO are in-memory buffers used to simulate files without writing to disk

# TextIOWrapper is a buffering translation layer that converts raw byte streams into text strings
    # Wraps binary streams (BytesIO) to read them as text with a specific encoding

# buffer: a temporary storage area in your computer's RAM that holds data while it is being moved from one place to another

In [ ]:
# StringIO
    # Text in memory
    # Use when you already have data as a standard Python string (like a hardcoded JSON block or text API response) and need to pass it to a function that expects a file object (e.g. CSV)
    # By wrapping a string inside io.StringIO, we are creating an in-memory buffer that acts exactly like an open file, but lives entirely in RAM
    # ** Avoid for massive datasets - can cause an Out-Of-Memory crash. Use BytesIO generators or chunks instead

# BytesIO
    # Bytes in memory
    # Use when extracting data from cloud storage or scraping binary files - these services return raw bytes, not strings

# TextIOWrapper
    # The bridge
    # Use when you have a binary stream (BytesIO) but your ETL tool insists on reading text strings
    # Decodes bytes on the fly

In [ ]:
# StringIO - CSV
import io
import csv

mock_csv_string1 = "name,age\nAlice,30\nBob,25"
mock_csv_string2 = """name,age,gender
Angela,32,F
Bill,61,M
Caleb,16,M
Diana,18,F
"""

csv_buffer1 = io.StringIO(mock_csv_string1)
csv_buffer2 = io.StringIO(mock_csv_string2)

reader1 = csv.DictReader(csv_buffer1)
for line in reader1:
    print(line)

reader2 = csv.DictReader(csv_buffer2)
for line in reader2:
    print(line)

{'name': 'Alice', 'age': '30'}
{'name': 'Bob', 'age': '25'}
{'name': 'Angela', 'age': '32', 'gender': 'F'}
{'name': 'Bill', 'age': '61', 'gender': 'M'}
{'name': 'Caleb', 'age': '16', 'gender': 'M'}
{'name': 'Diana', 'age': '18', 'gender': 'F'}


In [4]:
# StringIO - JSON
import io
import json

mock_json_string = '{"pipeline_status": "success", "records": 100}'
json_buffer = io.StringIO(mock_json_string)

data = json.load(json_buffer)
print(data)

{'pipeline_status': 'success', 'records': 100}


In [5]:
# StringIO: CSV --> JSON
import csv
import json
import io 

# 1. EXTRACT
mock_csv_file_data = """employee_id,name,department,salary
101,Alice Smith,Engineering,110000
102,Bob Jones,Marketing,85000
103,Charlie Brown,Engineering,95000
104,Diana Prince,HR,70000
"""

print("--- Step 1: Create in-memory buffer ---")
csv_buffer = io.StringIO(mock_csv_file_data)
print(f"Buffer created")


# 2. TRANSFORM
print("--- Step 2: Transform data from buffer ---")
reader = csv.DictReader(csv_buffer)
transformed_records = []

for row in reader:
    if row["department"] == "Engineering":
        clean_name = row["name"].strip().upper()
        numeric_salary = int(row["salary"])
        transformed_row = {
            "id": row.get("employee_id"),
            "employee_name": clean_name,
            "annual_salary": numeric_salary
        }
        transformed_records.append(transformed_row)

print(f"Transformed {len(transformed_records)} records")


# 3. LOAD
print("--- Step 3: Load data to final JSON string ---")
final_json_output = json.dumps(transformed_records, indent=4)

print(final_json_output)

--- Step 1: Create in-memory buffer ---
Buffer created
--- Step 2: Transform data from buffer ---
Transformed 2 records
--- Step 3: Load data to final JSON string ---
[
    {
        "id": "101",
        "employee_name": "ALICE SMITH",
        "annual_salary": 110000
    },
    {
        "id": "103",
        "employee_name": "CHARLIE BROWN",
        "annual_salary": 95000
    }
]


# Other

In [9]:
scores = [85,92,71,98,64,77,24,41,100]

has_failing_grade = any(score < 50 for score in scores)
all_passed = all(score >= 50 for score in scores)

print(f"has_failing_grade: {has_failing_grade}")
print(f"all_passed: {all_passed}")

has_failing_grade: True
all_passed: False


In [16]:
words = ["apple", "strawberry", "banana", "orange", "carrot"]
words.sort(key=lambda x: len(x))

print(words)

['apple', 'banana', 'orange', 'carrot', 'strawberry']


In [17]:
keys = ["name", "age", "role"]
values = ["Devon", 28, "Engineer"]
user_dict = dict(zip(keys,values))

print(user_dict)

{'name': 'Devon', 'age': 28, 'role': 'Engineer'}


In [18]:
text = "the quick brown fox jumped over the lazy dog"
new_text = text.replace("quick", "fast").replace("jumped", "leapt")

print(new_text)

the fast brown fox leapt over the lazy dog


In [19]:
print(new_text.endswith("dog"))

True


# Another ETL - generator

In [36]:
import csv
from datetime import datetime
import io
import logging

logging.basicConfig(level=logging.ERROR)

csv_data = """transaction_id,customer_id,customer_name,transaction_date,amount,currency,product
1,2,bob,2026-08-01,14.32,USD,baseball
2,3,Kate,2026-07-31,17.99,CAD,shirt
3,4,CinDy,2026-05-24,25.00,NZD,toy
4,5,PEter,2026-09-01,11.29,USD,pants
5,6,Mildred,2026-06-14,34.00,AUD,swimsuit
"""

csv_file_path = io.StringIO(csv_data)
print(csv_file_path)

In [30]:
def extract(file_path):
    print("--- Extracting ---")
    reader = csv.DictReader(file_path)
    for row in reader:
        print(row)
        yield row

In [31]:
def transform(records):
    print("--- Transforming ---")
    curr_time = datetime.now()

    for row in records:
        row = {
            "transaction_id": row.get("transaction_id"),
            "customer_id": row.get("customer_id"),
            "customer_name": row.get("customer_name"),
            "transaction_date": row.get("transaction_date"),
            "price": float(row.get("amount")),
            "currency": row.get("currency"),
            "is_usd": "True" if row.get("currency") == "USD" else "False",
            "product": row.get("product"),
            "processing_time": datetime.strftime(curr_time, "%Y-%m-%dT%H:%M:%S")
        }

        yield row

In [ ]:
def load_data(transformed_records, outfile):
    print(f"Loading data to {outfile}")
    first_row = next(transformed_records, None)
    field_names = first_row.keys()

    with open(outfile, mode="w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=field_names)
        writer.writeheader()
        
        for row in transformed_records:
            writer.writerow(row)

In [40]:
raw_stream = extract(csv_file_path)
cleaned_stream = transform(raw_stream)
load_data(cleaned_stream, "../output/etl_test_csv.csv")

Loading data to ../output/etl_test_csv.csv


TypeError: 'generator' object is not subscriptable